In [9]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -o ml-latest-small.zip


Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [10]:
%pip install -q pandas scikit-learn
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity



In [11]:
ratings = pd.read_csv('ml-latest-small/ratings.csv', usecols=['userId','movieId','rating'])
movies  = pd.read_csv('ml-latest-small/movies.csv',  usecols=['movieId','title','genres'])


In [12]:
user_item = ratings.pivot_table(index='userId', columns='movieId', values='rating', fill_value=0.0)
item_sim = cosine_similarity(user_item.T)
item_sim_df = pd.DataFrame(item_sim, index=user_item.columns, columns=user_item.columns)


In [13]:
def recommend_similar_by_title(title, top_n=10):
    row = movies[movies['title'] == title]
    if row.empty:
        return []
    mid = int(row.iloc[0]['movieId'])
    if mid not in item_sim_df:
        return []
    sims = item_sim_df[mid].sort_values(ascending=False)
    top_ids = [i for i in sims.index if i != mid][:top_n]
    return movies[movies['movieId'].isin(top_ids)][['movieId','title']].reset_index(drop=True)


In [14]:
recommend_similar_by_title("Toy Story (1995)", top_n=5)


,movieId,title
0,260,Star Wars: Episode IV - A New Hope (1977)
1,356,Forrest Gump (1994)
2,480,Jurassic Park (1993)
3,780,Independence Day (a.k.a. ID4) (1996)
4,3114,Toy Story 2 (1999)
